# 01. Eventos de tarefa: rótulo e unidade de análise

**Artigo:** Predição de Falhas em Tarefas em Ambientes de Computação em Nuvem
**Base:** Google Cluster Data 2011 (clusterdata-2011-2), tabela `task_events`

Este notebook:
1. lê os 500 arquivos da tabela `task_events` diretamente do bucket público do Google;
2. seleciona uma amostra de cerca de 10% dos jobs;
3. define o rótulo (FAIL = falha, FINISH = conclusão) e a unidade de análise;
4. grava `saida/primeira_tentativa.parquet`, usado no notebook 02.

Tipos de evento no trace: 0 = SUBMIT, 1 = SCHEDULE, 2 = EVICT, 3 = FAIL, 4 = FINISH, 5 = KILL, 6 = LOST.

## 1. Configuração do ambiente

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
import numpy as np
import pandas as pd

# Pasta de trabalho no Google Drive (dados intermediários e figuras)
BASE = '/content/drive/MyDrive/artigo_predicao'
for d in ['ev_parts', 'uso_parts', 'saida', 'figuras']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

KEY = ['job_id', 'task_index']   # identificador de uma tarefa no trace
FRAC = 10                        # amostra: jobs com job_id % 10 == 0 (~10% dos jobs)

## 2. Conferência do esquema das tabelas
Os arquivos do trace não têm cabeçalho; a ordem das colunas vem do `schema.csv` publicado com os dados.

In [ ]:
schema = pd.read_csv('https://storage.googleapis.com/clusterdata-2011-2/schema.csv')
print(schema.to_string())

## 3. Leitura e filtragem da tabela `task_events`
Cada arquivo é lido apenas com as colunas necessárias, filtrado para a amostra de jobs e para os eventos de interesse (SUBMIT e eventos terminais), e salvo em parquet no Drive. Instantes iguais a 0 (antes do início da coleta) ou a 2⁶³ − 1 (após o fim) são descartados. Se a sessão cair, basta rodar de novo: arquivos já processados são pulados.

In [ ]:
URL_EV = 'https://storage.googleapis.com/clusterdata-2011-2/task_events/part-{:05d}-of-00500.csv.gz'
COLS_EV = ['time','missing_info','job_id','task_index','machine_id','event_type','user',
           'scheduling_class','priority','cpu_req','ram_req','disk_req','diff_machine']
USE_EV = ['time','job_id','task_index','event_type','scheduling_class','priority',
          'cpu_req','ram_req','disk_req']
TIPOS_EV = {'time':'int64','job_id':'int64','task_index':'int32','event_type':'int8',
            'scheduling_class':'Int8','priority':'Int8',
            'cpu_req':'float32','ram_req':'float32','disk_req':'float32'}
TMAX = 2**63 - 1

def processa_eventos(i):
    out = f'{BASE}/ev_parts/p{i:03d}.parquet'
    if os.path.exists(out):
        return
    df = pd.read_csv(URL_EV.format(i), header=None, names=COLS_EV, usecols=USE_EV, dtype=TIPOS_EV)
    df = df[(df.job_id % FRAC == 0) & df.event_type.isin([0, 2, 3, 4, 5, 6])]
    df = df[(df.time > 0) & (df.time < TMAX)]
    df.to_parquet(out, index=False)

### 3.1 Teste com 3 arquivos

In [ ]:
for i in range(3):
    processa_eventos(i)
t = pd.concat([pd.read_parquet(f'{BASE}/ev_parts/p{i:03d}.parquet') for i in range(3)])
print(t.shape)
print(t.event_type.value_counts())
t.head()

### 3.2 Processamento dos 500 arquivos

In [ ]:
for i in range(500):
    for tentativa in range(3):
        try:
            processa_eventos(i)
            break
        except Exception as e:
            print(i, e)
            time.sleep(10)

## 4. Escolha da unidade de análise
Uma tarefa que falha pode ser ressubmetida e acumular vários eventos terminais. Três alternativas foram avaliadas:

| Alternativa | Descrição | Situação |
|---|---|---|
| A | tarefas com um único evento terminal | descartada: elimina quase todas as tarefas com falha |
| B | cada tentativa (execução) como observação | descartada: poucas tarefas ressubmetidas dominam a classe de falha |
| C | **primeira tentativa de cada tarefa** | **adotada** |

As células abaixo reproduzem o diagnóstico que sustentou a decisão.

In [ ]:
ev = pd.concat([pd.read_parquet(f'{BASE}/ev_parts/p{i:03d}.parquet') for i in range(500)],
               ignore_index=True).drop_duplicates()
ATTRS = ['scheduling_class','priority','cpu_req','ram_req','disk_req']

# todos os eventos terminais, em ordem temporal dentro de cada tarefa
term = ev[ev.event_type.isin([2, 3, 4, 5, 6])].sort_values(KEY + ['time']).copy()

### 4.1 Alternativa A: tarefas com um único evento terminal

In [ ]:
n_term = term.groupby(KEY).size().rename('n_term').reset_index()
t1 = term.merge(n_term, on=KEY)
alt_a = t1[(t1.n_term == 1) & t1.event_type.isin([3, 4])]
com_fail = term[term.event_type == 3][KEY].drop_duplicates()
print('tarefas com algum FAIL:', len(com_fail))
print('tarefas mantidas na alternativa A:', len(alt_a), '| falhas mantidas:', int((alt_a.event_type == 3).sum()))

### 4.2 Alternativa B: cada tentativa como observação

In [ ]:
term['t_ini'] = term.groupby(KEY).time.shift(1).fillna(0).astype('int64')   # fim da tentativa anterior
tent = term[term.event_type.isin([3, 4])].drop(columns=ATTRS).rename(columns={'time': 't_term'})
tent['falha'] = (tent.event_type == 3).astype('int8')
por_tarefa = tent.groupby(KEY).size()
print('tentativas:', len(tent), '| falhas:', int(tent.falha.sum()))
print(por_tarefa.describe().to_string())
top = por_tarefa[por_tarefa > 100].index
print('falhas vindas de tarefas com >100 tentativas:', int(tent.set_index(KEY).loc[top].falha.sum()))

### 4.3 Alternativa C (adotada): primeira tentativa de cada tarefa
A primeira tentativa é a que não tem evento terminal anterior (`t_ini == 0`). Os atributos de submissão vêm do último SUBMIT anterior ao término; tarefas sem SUBMIT correspondente são descartadas.

In [ ]:
sub = ev[ev.event_type == 0][KEY + ['time'] + ATTRS].sort_values('time')
prim = tent[tent.t_ini == 0].sort_values('t_term')
prim = pd.merge_asof(prim, sub, left_on='t_term', right_on='time', by=KEY,
                     direction='backward').drop(columns='time')
prim = prim[prim.priority.notna()].copy()
# dia do trace: os instantes contam a partir de 600 s antes do início da coleta
prim['dia'] = ((prim.t_term - 600_000_000) // 86_400_000_000 + 1).astype('int16')
prim.to_parquet(f'{BASE}/saida/primeira_tentativa.parquet', index=False)

n_tarefas = term[KEY].drop_duplicates().shape[0]
print('tarefas com evento terminal:', n_tarefas)
print('primeira tentativa FAIL/FINISH:', len(prim), '| falhas:', int(prim.falha.sum()), f'({prim.falha.mean():.4f})')
print('excluídas (1ª tentativa EVICT/KILL/LOST ou sem SUBMIT):', n_tarefas - len(prim))
print(prim.groupby('dia').falha.agg(['size', 'mean']).to_string())
print('teste (dias 24-29):', prim[prim.dia >= 24].falha.agg(['size', 'mean']).to_dict())

## 5. Resultados obtidos (execução de 28/09/2026, amostra de 10% dos jobs)

| Medida | Valor |
|---|---|
| Tarefas com algum FAIL | 133.021 |
| Alternativa A: falhas mantidas | 774 (99,4% das tarefas com falha excluídas) |
| Alternativa B: tentativas / falhas | 4.893.001 / 3.036.753 |
| Alternativa B: falhas vindas de tarefas com >100 tentativas | 2.500.983 (82%) |
| Alternativa C: tarefas / falhas | 1.821.654 / 100.730 (5,53%) |
| Alternativa C: excluídas | 114.965 |
| Alternativa C: teste (dias 24 a 29) | 349.282 tarefas, 4,04% de falha |

Na alternativa C, os dias 2 e 10 (que Santos, 2023, excluía do treino por concentrarem falhas) têm taxa de falha de 1,3% e 2,5%, abaixo da média: a concentração observada na literatura vem das ressubmissões.